Install Packages

In [2]:
! pip install langchain langchain-huggingface chromadb sentence-transformers

Defaulting to user installation because normal site-packages is not writeable


In [3]:
from langchain_text_splitters import RecursiveCharacterTextSplitter 
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

/home/atul/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
documents = [
    """
    The company leave policy states that full-time employees can carry forward up to 10 unused leave days.
    Contractors are not allowed to carry forward unused leave days.
    """,
    """
    The refund policy states that annual subscriptions are refundable within 30 days of purchase.
    Monthly subscriptions are non-refundable.
    """,
    """
    The work from home policy allows employees to work remotely up to three days per week.
    Contractors need manager approval before working remotely.
    """
]


In [5]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 200,
    chunk_overlap = 30
)

splits = text_splitter.create_documents(documents)

In [6]:
for i, doc in enumerate(splits):
    print(f"Chunks {i+1}: ")
    print(doc.page_content)
    print("-"*50)

Chunks 1: 
The company leave policy states that full-time employees can carry forward up to 10 unused leave days.
    Contractors are not allowed to carry forward unused leave days.
--------------------------------------------------
Chunks 2: 
The refund policy states that annual subscriptions are refundable within 30 days of purchase.
    Monthly subscriptions are non-refundable.
--------------------------------------------------
Chunks 3: 
The work from home policy allows employees to work remotely up to three days per week.
    Contractors need manager approval before working remotely.
--------------------------------------------------


In [7]:
embedding_model = HuggingFaceEmbeddings(
    model_name = "BAAI/bge-small-en-v1.5",
    encode_kwargs = {"normalize_embeddings": True}
)

/home/atul/.local/lib/python3.10/site-packages/torch/cuda/__init__.py:180: UserWarning: CUDA initialization: The NVIDIA driver on your system is too old (found version 12080). Please update your GPU driver by downloading and installing a new version from the URL: http://www.nvidia.com/Download/index.aspx Alternatively, go to: https://pytorch.org to install a PyTorch version that has been compiled with your version of the CUDA driver. (Triggered internally at /pytorch/c10/cuda/CUDAFunctions.cpp:119.)
  return torch._C._cuda_getDeviceCount() > 0
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 13796.60it/s]
BertModel LOAD REPORT from: BAAI/bge-small-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [8]:
vectorstore = Chroma.from_documents(
    documents=splits,
    embedding = embedding_model
)

In [9]:
retriever = vectorstore.as_retriever(
    search_kwargs = {"k":2}
)

In [10]:
query = "Can contractors carry forward unused leave?"
retrieved_docs = retriever.invoke(query)

for i,doc in enumerate(retrieved_docs):
    print(f"Retrived {i+1}:")
    print(doc.page_content)
    print("-"*50)

Retrived 1:
The company leave policy states that full-time employees can carry forward up to 10 unused leave days.
    Contractors are not allowed to carry forward unused leave days.
--------------------------------------------------
Retrived 2:
The work from home policy allows employees to work remotely up to three days per week.
    Contractors need manager approval before working remotely.
--------------------------------------------------


In [11]:
query = "Do contractors need approval for remote work?"

retrieved_docs = retriever.invoke(query)

for i,doc in enumerate(retrieved_docs):
    print(f"Retrived {i+1}:")
    print(doc.page_content)
    print("-"*50)

Retrived 1:
The work from home policy allows employees to work remotely up to three days per week.
    Contractors need manager approval before working remotely.
--------------------------------------------------
Retrived 2:
The company leave policy states that full-time employees can carry forward up to 10 unused leave days.
    Contractors are not allowed to carry forward unused leave days.
--------------------------------------------------


In [12]:
! ollama pull qwen2.5:1.5b

pulling manifest ⠙ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest ⠧ pulling manifest ⠇ pulling manifest ⠋ pulling manifest 
pulling 183715c43589: 100% ▕██████████████████▏ 986 MB                         
pulling 66b9ea09bd5b: 100% ▕██████████████████▏   68 B                         
pulling eb4402837c78: 100% ▕██████████████████▏ 1.5 KB                         
pulling 832dd9e00a68: 100% ▕██████████████████▏  11 KB                         
pulling 377ac4d7aeef: 100% ▕██████████████████▏  487 B                         
verifying sha256 digest 
writing manifest 
success 


In [13]:
! pip install langchain-ollama

Defaulting to user installation because normal site-packages is not writeable


In [14]:
from langchain_ollama import ChatOllama

llm= ChatOllama(
    model = "qwen2.5:1.5b",
    temperature=0
)

In [23]:
query = "Can contractors carry forward unused leave?"

retrieved_docs = retriever.invoke(query)

for i, doc in enumerate(retrieved_docs):
    print(f"Retrieved {i+1}:")
    print(doc.page_content)
    print("-" * 50)


Retrieved 1:
The company leave policy states that full-time employees can carry forward up to 10 unused leave days.
    Contractors are not allowed to carry forward unused leave days.
--------------------------------------------------
Retrieved 2:
The work from home policy allows employees to work remotely up to three days per week.
    Contractors need manager approval before working remotely.
--------------------------------------------------


In [24]:
context = "\n\n".join([doc.page_content for doc in retrieved_docs])

prompt = f"""
You are answering questions using ONLY the context.

Context:
{context}

Question:
{query}

Instructions:
- If the context directly contains the answer, answer it.
- Do not say "I don't know" if the context contains the answer.
- If contractors are mentioned in the context, use that information.

Answer in one short sentence:
"""


In [25]:
response = llm.invoke(prompt)
print(response.content)


No, contractors cannot carry forward unused leave days.


In [28]:
def ask_rag(query):
    retrived_docs = retriever.invoke(query)
    context = "\n\n".join([doc.page_content for doc in retrieved_docs])

    prompt = f"""
you are answering questions using ONLY the context.

Context:{context}

Question:{query}

Instructions:
- If the context directly contains the answer, answer it.
- If the answer is not in the context, say you don't know.
- Answer clearly and briefly.

Answer:
"""
    
    response = llm.invoke(prompt)
    return response.content

In [29]:
ask_rag("Can contractors carry forward unused leave?")


'No, contractors are not allowed to carry forward unused leave days.'

In [30]:
ask_rag("Can I get a refund for an annual subscription?")


"I don't know."

In [31]:
ask_rag("Do contractors need approval for remote work?")


'Yes, contractors need manager approval before working remotely.'